In [ ]:
import torchvision
from torchvision import transforms

transform1=transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip()
    ]
)

transform2=transforms.ToTensor()



transform= transforms.ToTensor()

train_data=torchvision.datasets.CIFAR10(root="/.data",train=True,download=True,transform=transform1)
test_data=torchvision.datasets.CIFAR10(root="/.data",train=False,download=True,transform=transform2)


  2%|▏         | 3.60M/170M [00:42<33:06, 84.0kB/s]

In [ ]:
from torch.utils.data import DataLoader

train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
test_loader=DataLoader(test_data,batch_size=32,shuffle=False)

In [ ]:
import matplotlib.pyplot as plt
for images,labels in train_loader:
    print(images.size())
    print(labels)

    for image in images:
        plt.imshow(image.permute(1,2,0))  # permute for changing [c,h,w] to [h,w,c].
        plt.show()
    break

In [ ]:
train_data.class_to_idx

In [ ]:
import torch.nn as nn
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.feature=nn.Sequential(
            nn.Conv2d(in_channels=3,out_channels=40,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=40,out_channels=60,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
         )

        self.linear=nn.Sequential(
            nn.Flatten(),
            nn.Linear(60*8*8,500),
            nn.ReLU(),
            nn.Linear(500,200),
            nn.ReLU(),
            nn.Linear(200,10),
            nn.ReLU() )

    def forward(self,x):
        x=self.feature(x)
        x=self.linear(x)
        return x



model=MyModel()


In [ ]:
for params in model.parameters():
    print(params.shape)

In [ ]:
import torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model=model.to(device)

In [ ]:
optimizer=torch.optim.Adadelta(model.parameters(),lr=0.001)
criterion=nn.CrossEntropyLoss()

In [ ]:
# # for 1 image
# for image,label in train_loader:
#     image=image.to(device)
#     y_pred=model(image)
#     print(y_pred)

#     break

In [ ]:
# now training
total_loss_per_batch=[]
for i in range (100):
    total_loss=0
    for image,label in train_loader:
        image=image.to(device)
        label=label.to(device)
        optimizer.zero_grad()
        y_pred=model(image)
        loss=criterion(y_pred,label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss=total_loss/len(train_loader)
    total_loss_per_batch.append(avg_loss)
    print(avg_loss)




